In [1]:
import pandas as pd 
import seaborn as sns
from datetime import datetime
from sqlalchemy import create_engine
from utils.config import *
from utils.db import *
import logging 
import os

In [2]:
engine = make_engine(DB_CONFIG)
test_connection(engine)

OperationalError: (pymysql.err.OperationalError) (1049, "Unknown database 'dld'")
(Background on this error at: https://sqlalche.me/e/20/e3q8)

In [3]:
# đoc csv 
df_raw = pd.read_csv('data/raw/dld_sales_transactions.csv')
df_raw.head()

C:\Users\ASUS\AppData\Local\Temp\ipykernel_5884\699360104.py:2: DtypeWarning: Columns (0: has_parking, 1: is_pre_registration) have mixed types. Specify dtype option on import or set low_memory=False.
  df_raw = pd.read_csv('dld_sales_transactions.csv')


,transaction_id,instance_date,reg_type_label,area_id,area_name_en,project_name,master_project_name,building_name,property_type_label,property_sub_type,usage_label,rooms_bucket,has_parking,area_sqm,price_aed,price_psf,procedure_id,procedure_name_en,is_pre_registration,quality_flags
0,1-11-1975-2,1975-01-01,Ready,232.0,Mirdif,NaN,NaN,NaN,Villa,NaN,Residential,NaN,False,692.31,50000,6.71,11.0,Sell,False,16
1,1-11-1990-1,1990-02-12,Ready,268.0,Al Musalla (Dubai),NaN,NaN,NaN,Land,NaN,Residential,NaN,False,232.26,1300000,520.00,11.0,Sell,False,0
2,1-11-1992-1,1992-04-20,Ready,270.0,Al Safaa,NaN,NaN,NaN,Land,NaN,Residential,NaN,False,929.03,500000,50.00,11.0,Sell,False,0
3,1-11-1993-1,1993-01-27,Ready,231.0,Al Mamzer,NaN,NaN,NaN,Villa,NaN,Residential,NaN,False,1189.16,2175000,169.92,11.0,Sell,False,0
4,1-11-1997-62,1997-09-08,Ready,271.0,Al Karama,NaN,NaN,NaN,Building,NaN,Commercial,NaN,False,422.71,2350000,516.48,11.0,Sell,False,0


In [7]:
df_raw.shape

(1372277, 20)

In [5]:
df_raw.info()

<class 'pandas.DataFrame'>
RangeIndex: 1372277 entries, 0 to 1372276
Data columns (total 20 columns):
 #   Column               Non-Null Count    Dtype  
---  ------               --------------    -----  
 0   transaction_id       1372277 non-null  str    
 1   instance_date        1372277 non-null  str    
 2   reg_type_label       1372277 non-null  str    
 3   area_id              1372149 non-null  float64
 4   area_name_en         1372149 non-null  str    
 5   project_name         1088508 non-null  str    
 6   master_project_name  1129462 non-null  str    
 7   building_name        973945 non-null   str    
 8   property_type_label  1372277 non-null  str    
 9   property_sub_type    1174322 non-null  str    
 10  usage_label          1372277 non-null  str    
 11  rooms_bucket         1092623 non-null  float64
 12  has_parking          1333380 non-null  object 
 13  area_sqm             1372276 non-null  float64
 14  price_aed            1372277 non-null  int64  
 15  price_psf

In [9]:
# cot nao null
df_raw.isnull().sum()

transaction_id              0
instance_date               0
reg_type_label              0
area_id                   128
area_name_en              128
project_name           283769
master_project_name    242815
building_name          398332
property_type_label         0
property_sub_type      197955
usage_label                 0
rooms_bucket           279654
has_parking             38897
area_sqm                    1
price_aed                   0
price_psf                 184
procedure_id            23764
procedure_name_en       23764
is_pre_registration     23764
quality_flags               0
dtype: int64

In [ ]:
# xu ly du lieu null
df_raw['area_name_en'].fillna("Unknown",inplace=True)

In [ ]:
logging.basicConfig(
    level = logging.INFO,
    format = '%(asctime)s | %(levelname)s | %(message)s',
    datefmt = '%H:%M:%S',
    handlers= [
        logging.FileHandler(
            f"log/ETL_pipeline_advanced_{datetime.now():%Y%m%d}.log",
            encoding='utf-8'
        ),
        logging.StreamHandler()      
    ],
    force=True
)

log = logging.getLogger(__name__)

def extract():
        log.info("EXTRACT start")
        df_a = pd.read_csv('Data/ad_metrics_2.csv')
        log.info(f"Get data ad done: {len(df_a)} rows")

        df_e = pd.read_csv('Data/employees.csv')
        log.info(f"Get data ad done: {len(df_e)} rows")

        log.info("Extract done")
        return df_a, df_e

def transform(df_a,df_e):
        log.info("TRANSFORM start")

        df_a = df_a.copy()
        df_a['date'] = pd.to_datetime(df_a['date'])
        df_a['cost_by_click'] = df_a['ad_spend']/df_a['clicks']
        df_a['dtime_data_created'] = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
        log.info(f"Trasform data ad done")

        df_e = df_e.copy() #SettingWithCopyWarning 
        df_e['enroll_date']=pd.to_datetime(df_e['enroll_date'])
        df_e['full_name']=df_e['full_name'].fillna("Unknown")
        df_e['fee_paid']=df_e['fee_paid'].fillna(df_e['fee_paid'].median())
        df_e['enroll_month']=df_e['enroll_date'].dt.month
        df_e['dtime_data_created'] = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
        log.info(f"Trasform data employee done")

        log.info("TRANSFORM done")

        return df_a, df_e

def load(df_a, df_e):
        log.info("LOAD start")
        engine=make_engine(DB_CONFIG)

        df_a.to_sql("fact_ad",engine, if_exists="append",index=False)
        df_e.to_sql("fact_employee",engine,if_exists="append",index=False)
        log.info("LOAD done")
